# Provenance Agent - Workflow Demo

The demo notebook for the provenance agent. 

Three parts:

1. **Software** - how a notebook's imported libraries become a
   citation-metadata DataFrame.
2. **Data** - how a notebook's used datasets are detected  become a
   citation-metadata DataFrame.
3. **The agent layer** - the three ways to invoke both workflows.

**Neither workflow returns citations** Each *injects a cell* into the target notebook, and the citations are that cell's output when you run it. Dataset retrieval calls `get_bibtex()` on a
LiPD object living in the kernel, which this code cannot reach from outside,
so it writes the retrieval code into your notebook instead. 

Injection is **in place** unless you pass `output_path`, so every cell below that
mutates a notebook points at a throwaway copy.

## Setup

The package is installed (`pip install -e ".[dev,google]"`), so everything
imports by name from anywhere. Nothing goes on `sys.path`.

Parts 1 and 2 are fully offline and need no API key. Only `agent.run` and
`%provenance` in Part 3 call a model.

In [ ]:
from provenance_agent.notebook_io import (
    parse_notebook,
    extract_libraries,
    strip_ipython_directives,
    validate_libraries,
)
from provenance_agent.citations import collect_library_entries
from provenance_agent.dataset_detection import (
    detect_datasets,
    detect_datasets_with_diagnostics,
)
from provenance_agent.data import (
    build_retrieval_cell,
    filter_datasets,
    inject_retrieval_cells,
    generate_data_workflow,
)

---

# Part 1 - Software building blocks

The software side is a static `ast` scan. It works on a notebook that has never
been run, needs no kernel, and needs no model.

## 1.1 `parse_notebook` - scan a full notebook

Takes a path to a `.ipynb` and returns the **sorted list** of top-level library
names it imports. Passing `None` auto-detects the current notebook, which
requires `ipynbname` and a running kernel.

In [ ]:
parse_notebook('../fixtures/sample.ipynb')

## 1.2 `extract_libraries` - parse a snippet

Works on a raw string rather than a file. It handles both `import X` and
`from X.Y import Z`, and always returns the top-level package
(`matplotlib.pyplot` becomes `matplotlib`).

In [ ]:
code_snippet = """
import numpy as np
import pandas as pd
from matplotlib.pyplot import plt
"""
print(extract_libraries(code_snippet))

## 1.3 `strip_ipython_directives` - make a cell parseable

Paleoclimate notebooks are full of `%matplotlib inline` and `!pip install ...`,
which are not valid Python and would make `ast.parse` fail. These are stripped
before parsing. Whole-cell magics whose body is not Python (`%%bash`) are dropped
entirely.

In [ ]:
raw = """
%matplotlib inline
!pip install pyleoclim
import pyleoclim as pyleo
"""
print(strip_ipython_directives(raw))

## 1.4 The pipeline: parse, validate, collect

1. `parse_notebook` extracts the imported libraries.
2. `validate_libraries` splits a requested subset into found and not-found.
3. `collect_library_entries` merges the matching BibTeX from the packaged
   `Citations/` data into one DataFrame, deduped by DOI.

There is no rendering step. APA rendering was removed along with the LLM chain
behind it, so **the citations are the DataFrame**. A library with no entry
becomes a `note` row rather than disappearing, so gaps stay visible.

In [ ]:
libraries = parse_notebook('../examples/paleoPCAlite.ipynb')
print('imported:', libraries)

found, not_found = validate_libraries(['pandas', 'numpy', 'fake_lib'], libraries)
print('found:    ', found)
print('not found:', not_found)

In [ ]:
entries = collect_library_entries(found)
print(f'{len(entries)} citation entries')
entries

## 1.5 Against the real corpus

`notebooks/examples/` doubles as the detection corpus. Running the parser over it
exercises the scan against real scientific notebooks.

In [ ]:
corpus = [
    '../examples/C02_b_DA_with_individual_seasonality.ipynb',
    '../examples/comparing-simulated-reconstructed-climate/CMIP6_LMR.ipynb',
    '../examples/comparing-simulated-reconstructed-climate/data_from_esm_cloudcat.ipynb',
    '../examples/comparing-simulated-reconstructed-climate/spatial_snapshots_xarray_bonuses.ipynb',
    '../examples/comparing-simulated-reconstructed-climate/VICS_dashboard.ipynb',
    '../examples/comparing-simulated-reconstructed-climate/widget_primer.ipynb',
]

for path in corpus:
    print(f'{path.split("/")[-1]:<55} {parse_notebook(path)}')

---

# Part 2 - Data building blocks

The data side is a **deterministic** AST and data-flow analysis. It never calls a
model and never executes notebook code.

## 2.1 `detect_datasets` - which datasets were actually used

Returns `[variable, tool]` pairs. A source is reported only when its lineage
reaches a recognized analysis call, or when it produces a live terminal tabular
DataFrame. **A dataset that is loaded and then abandoned is not reported** - that
is what makes the tool cite what was *used*, and it is the behavior that
surprises people most.

Recognized sources are PyLiPD, PyleoTUPS, and LiPDGraph, plus `xarray` and
`pandas` loaders.

In [ ]:
detect_datasets('../examples/paleoPCAlite.ipynb')

### When it finds nothing, ask why

`detect_datasets_with_diagnostics` returns the same pairs plus an explanation for
every analysis call whose data it could not trace back to a source.

An empty result with **no** warnings means no analysis calls were found at all.
An empty result **with** warnings means analysis was found but could not be
connected to a recognized loader. Those are different answers.

In [ ]:
diagnostics = detect_datasets_with_diagnostics('../examples/paleoPCAlite.ipynb')
print('pairs:', diagnostics['pairs'])
for warning in diagnostics['warnings']:
    print('warning:', warning)

## 2.2 `build_retrieval_cell` - the source for one dataset

Each source gets a retrieval block that reuses the object already loaded in your
kernel. LiPDGraph is the special case: its terminal variable is a DataFrame, so
the block lifts the `dataSetName` column, loads those datasets into a fresh
`LiPD` object from the endpoint, and only then calls `get_bibtex()`.

In [ ]:
for variable, tool in [('D', 'PyLiPD'), ('ds', 'PyleoTUPS'), ('filtered_df2', 'LiPDGraph')]:
    print(f'# --- {tool} ---')
    print(build_retrieval_cell(variable, tool))
    print()

## 2.3 `filter_datasets` - narrow the detected pairs

Retains the older variable-level filtering alongside the current dataset-name
targeting. User-facing selection goes through `targets=`, which takes
**dataset names, never notebook variable names**.

In [ ]:
sample = [['D', 'PyLiPD'], ['ds', 'PyleoTUPS'], ['filtered_df2', 'LiPDGraph']]
print('all:      ', filter_datasets(sample))
print('LiPDGraph:', filter_datasets(sample, tool='LiPDGraph'))
print('just ds:  ', filter_datasets(sample, variable='ds'))

## 2.4 `inject_retrieval_cells` - append to a notebook node

**One cell for every source**, not one per dataset. Re-running a workflow
replaces its own cell rather than stacking a stale second one.

In [ ]:
import nbformat

demo = nbformat.v4.new_notebook()
inject_retrieval_cells(demo, [['filtered_df2', 'LiPDGraph']])
print(demo.cells[-1].source)

## 2.5 `generate_data_workflow` - end to end

Detect, filter, inject, write. Writing to `output_path` leaves the original
untouched. Open the copy and run the injected cell **in its own kernel**, where
`filtered_df2` exists, to get the BibTeX.

Use `targets=` to request specific datasets by name. Note that `variable=` is
retired at this layer and raises `ValueError`; a specific PyleoTUPS study also
cannot be targeted, because its names exist only inside the live object.

In [ ]:
pairs = generate_data_workflow(
    '../examples/paleoPCAlite.ipynb',
    output_path='../examples/paleoPCAlite_demo.ipynb',
)
print('injected a retrieval cell for:', pairs)

---

# Part 3 - The agent layer

Parts 1 and 2 call the workflow functions directly. The agent wraps that same
work in three layers, shown lowest to highest. They produce identical citations
and differ only in how you invoke them.

| Layer | Needs a model | Writes |
|---|---|---|
| `cite_software` / `cite_data` | no | `output_path`, or in place |
| `agent.run` | yes | **in place** |
| `%provenance` | yes | **in place** |

Because the top two write in place, everything below points at a throwaway copy.

In [ ]:
import shutil

SOURCE = '../examples/paleoPCAlite.ipynb'
DEMO = '../examples/paleoPCAlite_demo.ipynb'
shutil.copy(SOURCE, DEMO)
print('working against', DEMO)

## 3.1 The direct functions

`cite_software` returns the library names it built a cell for. `cite_data`
returns the `[variable, tool]` pairs it built a retrieval cell for. Neither
returns citations.

`cite_software` also takes the filters the natural-language layers expose through
language: `libraries=` for specific packages, `citation_types=` to keep only
`paper` or only `software` entries.

In [ ]:
from provenance_agent import cite_data, cite_software

print('software:', cite_software(SOURCE, output_path=DEMO))
print('software, filtered:', cite_software(
    SOURCE,
    libraries=['pyleoclim', 'pandas'],
    citation_types=['software'],
    output_path=DEMO,
))
print('data:', cite_data(SOURCE, output_path=DEMO))

## 3.2 The natural-language router - `agent.run`

Classifies the request into a typed decision, dispatches the workflows it
selected, and returns an envelope. **This is the only part of the whole tool that
calls a model**, and it needs an API key.

The envelope's `dispatch` key holds one `{name, args, result}` per tool called;
`verification` records what changed in the notebook, diffed before and after
without executing anything.

An unclear request, or one naming software the notebook does not import, comes
back as `status: "warning"` and **leaves the notebook unmutated**.

In [ ]:
from provenance_agent.agent import run

shutil.copy(SOURCE, DEMO)
result = run('cite the software', DEMO)

print('status:', result['status'])
for call in result['dispatch']:
    print(call['name'], '->', call['result'])
print('verification:', result['verification'])

## 3.3 The `%provenance` magic

The notebook-native layer, and the intended experience. The extension name is
`provenance`, **not** `provenance_agent`: the latter is the import name, and
`%load_ext provenance_agent` fails.

In [ ]:
%load_ext provenance

Auto-detection matches the running kernel against the Jupyter server's session
list, which VSCode does not expose the same way. In VSCode, setting the path
explicitly is the normal path rather than a fallback.

In [ ]:
shutil.copy(SOURCE, DEMO)
%provenance_notebook ../examples/paleoPCAlite_demo.ipynb

In [ ]:
%provenance cite the software

A named target is pulled out of the request and passed through as a filter.

In [ ]:
%provenance cite Pyleoclim

The same magic routes to the data workflow. Retrieval needs the dataset objects
live in the kernel, so this injects one retrieval cell covering every detected
source and reports what it wrote. The citations are that cell's output once you
reload the target and run it: each source leaves `_bib_{var}` and `_meta_{var}`
bound in the kernel, and the cell displays every metadata frame.

In [ ]:
%provenance cite the datasets

## A note on formats

Naming a format changes nothing. `fmt` is accepted and ignored everywhere:
`"bibtex"`, `"apa"`, and any other value produce byte-identical output, and no
value is rejected. APA rendering was removed with the LLM chain that produced it,
and the parameter is held open for a future non-LLM implementation that would
render from the metadata DataFrame.

## What to do next

Reload `../examples/paleoPCAlite_demo.ipynb` and run the cells the workflows
appended. Two cells is correct: one software, one data. Each workflow owns
exactly one.